In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.5 Eigenvalues, Diagonalization, and the SVD

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.5",
    title="Eigenvalues, Diagonalization, and the SVD",
    blurb="The spectral theorem, the generalized eigenproblem behind every "
    "normal mode, and the singular value decomposition — the master "
    "factorization that powers least squares, data compression, and "
    "the tensor networks of many-body quantum physics.",
    difficulty="advanced",
    estimate="130–170 min",
)

## Notebook overview

Eigenvalues we already know as the invariant directions of a linear map; what
we have probably *not* seen is how a computer finds them, and that it does so
without ever touching the characteristic polynomial, whose roots are far too
ill-conditioned to trust (a direct callback to the conditioning lessons of
[§0.1](floating-point.ipynb) and [§0.4](linear-systems.ipynb)). This notebook
builds the numerical eigenproblem from the well-behaved
symmetric case and the **spectral theorem**, through the **QR algorithm** that
`scipy.linalg.eigh` actually runs, to the **generalized eigenproblem** $K\mathbf
v=\lambda M\mathbf v$ that is the literal engine of every normal-mode calculation
in [§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb),
[§2.6](../02-classical-mechanics/rigid-body.ipynb), and
[§2.7](../02-classical-mechanics/small-oscillations.ipynb).

Size then changes the question. A matrix small enough to write down can simply be
handed to `eigh`, but the ones a physics computation actually produces are huge
and almost entirely zeros, and it is the zeros, not the arithmetic, that break the
calculation first. So the notebook also assembles a **sparse** finite-difference
Laplacian, writes a **Lanczos** iteration that never sees the matrix except
through products $A\mathbf x$, and derives the **shift-invert** trick that turns a
method biased toward the *largest* eigenvalues into the standard tool for the
smallest. This is the machinery that
[§3.4](../03-electrodynamics/laplace-poisson.ipynb) stores a boundary-value
problem with, that [§3.9](../03-electrodynamics/waveguides-cavities.ipynb) finds
waveguide cutoffs with, and that
[§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb) and
[§8.2](../08-electronic-structure/exact-laboratory.ipynb) find ground states with:
they use it, and this is where it is explained.

Its second half is the **singular value decomposition**: the one factorization
that works for *any* matrix, square or not, and arguably the most useful in all
of applied mathematics. We will see its geometry (every linear map is a rotation,
a scaling, and another rotation), prove the **Eckart–Young** theorem (the best
low-rank approximation is the truncated SVD), and watch it strip noise from a
nearly-low-rank matrix. That last move (keep the few large singular values,
discard the small tail) is, almost verbatim, how the **tensor networks** of
many-body quantum physics tame an exponentially large Hilbert space — a
connection this course names and motivates but nowhere implements.

Throughout we lean on [§0.4](linear-systems.ipynb): its condition number $\kappa$, its QR
factorization, and its "never invert" rule all return here. The only animation is
the SVD's geometric action, where motion genuinely shows what a still cannot; the
spectra and error plots are static.

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: a reconstruction $A=V\Lambda V^\top$, a known normal-mode
> frequency, the closed-form spectrum of a discrete Laplacian, an Eckart–Young
> error equal to $\sigma_{k+1}$. A ✓ is strong
> evidence; a ✗ is a prompt to *locate the discrepancy*, not a verdict.

> **Scope.** A working review, not a course in matrix computations. See Trefethen
> & Bau, *Numerical Linear Algebra* {cite}`trefethen1997` (the eig and SVD
> lectures) and Golub & Van Loan {cite}`golubvanloan`.

## Theory in brief

### The eigenproblem, and why not the characteristic polynomial

An eigenpair of a square matrix $A$ satisfies

```{math}
:label: eq-eigprob
A\mathbf v = \lambda\mathbf v,
```

so $\mathbf v$ is a direction the map only stretches, by the factor $\lambda$.
We learned to find $\lambda$ as the roots of $\det(A-\lambda I)=0$, but that is
precisely what a numerical eigensolver refuses to do: the map from a polynomial's
coefficients to its roots is wildly ill-conditioned (Wilkinson's classic warning),
so forming the characteristic polynomial and rooting it loses most of the digits.
Instead, eigenvalues are found by *iteration*: repeated orthogonal
transformations that drive the matrix toward triangular form.

### The symmetric case and the spectral theorem

The friendliest (and most physical) case is a real symmetric $A=A^\top$
(observables, quadratic forms, mass and stiffness matrices). Its eigenvalues are
real and its eigenvectors can be chosen orthonormal, so $A$ diagonalizes as

```{math}
:label: eq-spectral
A = V\Lambda V^\top = \sum_i \lambda_i\,\mathbf v_i\mathbf v_i^\top,
\qquad V^\top V = I,
```

the **spectral theorem**: $A$ is a weighted sum of orthogonal projectors onto its
eigendirections. `scipy.linalg.eigh` returns exactly this $V$ and $\Lambda$.

### The QR algorithm

How does `eigh` get there? By the **QR algorithm**, which reuses the QR
factorization of [§0.4](linear-systems.ipynb) in a strikingly simple loop:
factor, then multiply the factors back in the opposite order,

```{math}
:label: eq-qralg
A_k = Q_k R_k, \qquad A_{k+1} = R_k Q_k,
```

and repeat. Each $A_{k+1}=Q_k^\top A_k Q_k$ is an orthogonal similarity of $A$, so
the eigenvalues are preserved, and the iterates converge to (block-)triangular
form with the eigenvalues marching out onto the diagonal. (Why so simple a loop
converges is genuinely subtle; Trefethen & Bau, *Numerical Linear Algebra*,
Lectures 28–29, supply the convergence theory.)

### The generalized eigenproblem

Normal modes do not come as $A\mathbf v=\lambda\mathbf v$ but as

```{math}
:label: eq-genev
K\mathbf v = \lambda M\mathbf v,
```

with a stiffness matrix $K$ and a mass matrix $M\ne I$: exactly the form derived
in [§2.7](../02-classical-mechanics/small-oscillations.ipynb). It is solved by
`eigh(K, M)`, whose eigenvectors are $M$-orthonormal; the
$\lambda$ are the squared normal-mode frequencies. This is the engine under
[§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb),
[§2.6](../02-classical-mechanics/rigid-body.ipynb), and
[§2.7](../02-classical-mechanics/small-oscillations.ipynb).

### Sparsity, and what a matrix-vector product already knows

Everything above assumes the matrix fits in memory as a rectangle of numbers.
At the sizes physics reaches it does not, and it does not have to: a
finite-difference Laplacian, a tight-binding Hamiltonian and a spin chain in the
computational basis all couple each row to a handful of neighbours, so all but
$O(N)$ of their $N^2$ entries are zero. An $N\times N$ array of doubles costs
$8N^2$ bytes, a gigabyte just past $N=10^4$, and a dense factorization of it
costs $O(N^3)$ work on top; storing and factoring the zeros is what fails, long
before the physics does. Sparse formats (`scipy.sparse`) keep only the non-zeros,
and the eigensolvers built on them (`scipy.sparse.linalg.eigsh`) ask the operator
for one thing only: the product $A\mathbf x$.

What products alone can reveal is the **Krylov subspace** generated by a starting
vector $\mathbf b$,

```{math}
:label: eq-krylov
\mathcal K_k(A,\mathbf b)=\operatorname{span}\{\mathbf b,\,A\mathbf b,\,A^2\mathbf b,\,
\dots,\,A^{k-1}\mathbf b\},
```

reached in $k-1$ products. Its dimension $k$ is tiny beside $N$, yet for symmetric
$A$ the eigenvalues of $A$ restricted to $\mathcal K_k$ (the **Ritz values**) lock
onto the *extremes* of the spectrum long before they resolve anything in the
interior. The bias is not a defect but the whole point: a ground-state energy, a
lowest cutoff frequency, a slowest relaxation mode all live at an edge of the
spectrum, and never in the middle.

### The singular value decomposition

Eigenvalues need a square matrix; the **SVD** needs nothing of the sort. Every
$m\times n$ matrix factors as

```{math}
:label: eq-svd
A = U\Sigma V^\top, \qquad U^\top U = I,\ V^\top V = I,
```

with $\Sigma$ diagonal holding the **singular values** $\sigma_1\ge\sigma_2\ge
\cdots\ge 0$. Geometrically every linear map is therefore the same three acts:
rotate ($V^\top$), scale along axes ($\Sigma$), rotate ($U$), so a matrix sends
the unit sphere to an ellipsoid whose semi-axes are the $\sigma_i$. The singular
values are $\sigma_i=\sqrt{\lambda_i(A^\top A)}$, tying the SVD back to the
symmetric eigenproblem. (That every matrix admits this factorization is a
theorem, not an observation; Trefethen & Bau, *Numerical Linear Algebra*,
Lectures 4–5, prove it.)

### Low-rank approximation: Eckart–Young

Truncating the SVD to its $k$ largest singular values gives the provably best
rank-$k$ approximation of $A$, and the error is exactly the first discarded
singular value:

```{math}
:label: eq-eckart
\min_{\operatorname{rank}B\le k}\lVert A-B\rVert_2 = \sigma_{k+1},
\qquad B = \sum_{i=1}^{k}\sigma_i\,\mathbf u_i\mathbf v_i^\top.
```

This single fact is the mathematical core of data compression and of the
tensor-network methods of many-body quantum physics alike. (Trefethen & Bau,
*Numerical Linear Algebra*, Lecture 5, gives the short proof.)

## Setup

Imports and one display setting — this notebook's Setup defines no functions
at all. It holds NumPy, Matplotlib and SymPy, the four `scipy.linalg`
factorizations the notebook leans on (`eigh`, `eigvalsh`, `qr`, `svd`), the
three `scipy.sparse` constructors and the Krylov eigensolver `eigsh` that the
sparse exercises are measured against them, `time.perf_counter` for that
measurement, the `ecp` validation and animation helpers, and a four-digit print
precision so that spectra and reconstructions read cleanly. Everything the
notebook is *about* — the QR algorithm iterated from scratch, the generalized
eigensolve behind the double pendulum's normal modes, the sparse Laplacian and
the Lanczos iteration that hunts its spectrum, the SVD's three geometric acts,
the rank-$k$ truncations of Eckart–Young, and the noise-stripping of a
nearly-low-rank matrix — you build in the exercise where it is earned.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.linalg import eigh, svd, qr, eigvalsh
from scipy.sparse import diags, identity, kron
from scipy.sparse.linalg import eigsh

from ecp import validate
from ecp.animate import show

np.set_printoptions(precision=4, suppress=True)

## Exercise 1 — The symmetric eigenproblem and the spectral theorem

The cleanest entry point is a real symmetric matrix, where {eq}`eq-spectral`
promises real eigenvalues and orthonormal eigenvectors. Take the explicit matrix

$$
A = \begin{bmatrix} 2 & 1 & 0 \\ 1 & 3 & 1 \\ 0 & 1 & 2 \end{bmatrix},
$$

which is symmetric, so the spectral theorem applies.

1. Diagonalize it with `scipy.linalg.eigh`.
2. Verify both halves of {eq}`eq-spectral`: the eigenvectors are orthonormal
   ($V^\top V=I$) and $A$ is rebuilt as $V\Lambda V^\top$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(V1.T @ V1, np.eye(3), "eigenvectors are orthonormal (VᵀV=I)", atol=1e-10)
validate.close(V1 @ np.diag(w1) @ V1.T, A1, "A = VΛVᵀ (spectral theorem)", atol=1e-10)

## Exercise 2 — Why not the characteristic polynomial

It is worth seeing *why* numerical eigensolvers avoid the textbook route. For the
same matrix $A=\begin{bmatrix}2&1&0\\1&3&1\\0&1&2\end{bmatrix}$ of Exercise 1, we
compare two routes against an exact reference. Because the coefficients-to-roots
map is ill-conditioned (the same kind of amplification $\kappa$ measured in
[§0.4](linear-systems.ipynb)), the polynomial route loses accuracy even on this
benign $3\times3$.

1. Compute the exact eigenvalues symbolically (`sympy.Matrix.eigenvals`) as the
   reference.
2. Compute them numerically with `scipy.linalg.eigvalsh` (backward-stable).
3. Form the characteristic polynomial's coefficients (`numpy.poly`) and root them
   (`numpy.roots`); compare both errors against the reference.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    err_eigh < err_charpoly,
    "the QR-based eigensolver beats rooting the characteristic polynomial",
    f"eigh {err_eigh:.2e} < charpoly {err_charpoly:.2e}",
)

## Exercise 3 — The QR algorithm from scratch

To see how `eigh` actually converges, we implement the unshifted QR algorithm of
{eq}`eq-qralg` using the very QR factorization built in
[§0.4](linear-systems.ipynb). Take the explicit
symmetric matrix

$$
A = \begin{bmatrix} 4 & 1 & 0 \\ 1 & 3 & 1 \\ 0 & 1 & 2 \end{bmatrix}.
$$

1. Iterate $A_k=Q_kR_k$, $A_{k+1}=R_kQ_k$ with `scipy.linalg.qr`; each step is an
   orthogonal similarity, so the spectrum is preserved.
2. Track the off-diagonal norm decaying as the diagonal converges to the
   eigenvalues ({numref}`fig-eigsvd-qr-conv`).
3. Confirm the converged diagonal matches `scipy.linalg.eigvalsh`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    diag_final,
    np.sort(eigvalsh(A3)),
    "the QR algorithm converges to the eigenvalues",
    atol=1e-6,
)

## Exercise 4 — The generalized eigenproblem (callback to 2.7)

Normal modes arrive as $K\mathbf v=\lambda M\mathbf v$, {eq}`eq-genev`, with a
mass matrix that is not the identity. Take the small-angle double pendulum of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb) with equal bobs and
rods ($m=\ell=1$, $g=9.81$), whose mass and
stiffness matrices are

$$
M = m\ell^2\begin{bmatrix} 2 & 1 \\ 1 & 1 \end{bmatrix}, \qquad
K = mg\ell\begin{bmatrix} 2 & 0 \\ 0 & 1 \end{bmatrix}.
$$

1. Solve the generalized problem with `scipy.linalg.eigh(K, M)`; the eigenvalues
   are the squared normal-mode frequencies.
2. Confirm they reproduce the analytic doubles
   $\omega_\pm^2=(2\mp\sqrt2)\,g/\ell$ — the same modes obtained by direct
   integration in [§1.3](../01-elementary-mechanics/double-pendulum.ipynb) and
   by linearization in
   [§2.7](../02-classical-mechanics/small-oscillations.ipynb), here from one
   eigensolve.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.close(
    omega,
    omega_analytic,
    "generalized eig gives the double-pendulum modes √((2∓√2)g/ℓ)",
    rtol=1e-9,
)

## Exercise 5 — Sparse storage: the matrix that is mostly zeros

Every matrix so far has been small enough to write down, and that is the last
thing a real physics matrix is. Discretize $-\nabla^2$ on the unit square with the
five-point stencil of [§3.4](../03-electrodynamics/laplace-poisson.ipynb), on an
$n\times n$ grid of interior nodes with Dirichlet boundaries, and the unknowns
number $N=n^2$: a modest $100\times100$ grid already brings $N$ to $10^4$. The
operator separates, because the two directions each contribute the same
one-dimensional second difference, so it is the Kronecker sum

```{math}
:label: eq-fd-laplacian
A = D\otimes I_n + I_n\otimes D,
\qquad
D = \frac{1}{h^2}\operatorname{tridiag}(-1,\,2,\,-1),
\qquad h=\frac{1}{n+1},
```

with $D$ and $I_n$ of size $n\times n$ and $h$ the grid spacing. Each row of $A$
then carries a diagonal entry and at most four neighbour couplings, so the
non-zeros grow like $5N$ while the dense array that would hold them grows like
$8N^2$ bytes. The `scipy.sparse` constructors `diags`, `identity` and `kron`
assemble {eq}`eq-fd-laplacian` directly in a format that stores only those
non-zeros, never the zeros between them.

The assembly can be graded exactly, because this operator's spectrum is known in
closed form: the eigenvectors are the sine products $\sin(p\pi x)\sin(q\pi y)$ and
the discrete eigenvalues are

```{math}
:label: eq-fd-spectrum
\lambda_{pq} = \frac{4}{h^2}\left[\sin^2\!\frac{p\pi h}{2} + \sin^2\!\frac{q\pi h}{2}\right],
\qquad p,q = 1,\dots,n,
```

which reduces to the continuum $\pi^2(p^2+q^2)$ as $h\to0$.

1. Write `laplacian_2d(n)` returning the operator of {eq}`eq-fd-laplacian` in CSR
   format together with $h$: build $D$ with `scipy.sparse.diags` (values
   $-1,2,-1$ on offsets $-1,0,+1$, scaled by $1/h^2$), take the two
   `scipy.sparse.kron` products against `scipy.sparse.identity(n)`, add them, and
   finish with `.tocsr()`. Call it at $n=20$ and report `A.nnz` and the density
   $\mathrm{nnz}/N^2$.
2. Certify the assembly: form the dense array with `.toarray()`, take its
   spectrum with `scipy.linalg.eigvalsh`, and compare against {eq}`eq-fd-spectrum`
   evaluated on the full $(p,q)$ grid.
3. Confirm the structure exactly. An $n\times n$ grid has $5n^2-4n$ non-zeros:
   five per row, less the one missing neighbour for each node on each of the four
   grid edges.
4. For $n=10,20,40,80,160,320$ compare the dense cost $8N^2$ bytes against the
   bytes CSR actually allocates
   (`A.data.nbytes + A.indices.nbytes + A.indptr.nbytes`), and plot both against
   $N$ ({numref}`fig-eigsvd-sparse-memory`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    lam_dense,
    lam_analytic,
    "the sparse assembly reproduces the analytic spectrum of −∇²",
    rtol=1e-10,
)
validate.check(
    A5.nnz == nnz_expected,
    "the five-point stencil gives exactly 5n²−4n non-zeros, never more than five per row",
    f"nnz = {A5.nnz}, 5n²−4n = {nnz_expected}",
)
validate.check(
    bytes_dense[-1] / bytes_sparse[-1] > 1000,
    "at n = 320 the discarded zeros would have cost over 1000× the matrix itself",
    f"{bytes_dense[-1] / 1e9:.1f} GB dense vs {bytes_sparse[-1] / 1e6:.1f} MB sparse "
    f"({bytes_dense[-1] / bytes_sparse[-1]:.0f}×)",
)

## Exercise 6 — Krylov subspaces: Lanczos, and shift-invert for the smallest

The dense spectrum taken in Exercise 5 was affordable only because $N=400$; the
same call at $n=320$ would ask for 84 GB and $O(N^3)$ work for numbers we do not
want. A Krylov method, {eq}`eq-krylov`, asks the operator for nothing but products
$A\mathbf x$. For symmetric $A$ the **Lanczos** iteration builds an orthonormal
basis $\mathbf q_1,\mathbf q_2,\dots$ of $\mathcal K_k(A,\mathbf q_1)$ with a
three-term recurrence,

```{math}
:label: eq-lanczos
A\mathbf q_j = \beta_{j-1}\mathbf q_{j-1} + \alpha_j\mathbf q_j + \beta_j\mathbf q_{j+1},
\qquad \alpha_j = \mathbf q_j^\top A\mathbf q_j,
\qquad \beta_j = \lVert\mathbf w_j\rVert,
```

where $\mathbf w_j$ is what remains of $A\mathbf q_j$ once the two previous basis
vectors are projected out, and $\mathbf q_{j+1}=\mathbf w_j/\beta_j$. Only three
terms appear because symmetry forces the projected matrix
$T_k=Q_k^\top A Q_k$ to be **tridiagonal**, with the $\alpha_j$ on the diagonal
and the $\beta_j$ beside it. Its eigenvalues, the Ritz values, then cost almost
nothing: a $k\times k$ symmetric tridiagonal eigensolve, with $k\ll N$.

In exact arithmetic the recurrence keeps the $\mathbf q_j$ orthogonal by itself.
In floating point it does not: as a Ritz value converges, rounding leaks its
eigendirection back into the basis and the iteration reports the same eigenvalue
several times over, the "ghost" eigenvalues, a loss of orthogonality of exactly
the kind [§0.1](floating-point.ipynb) warns about. Subtracting the stored basis
off once per step (full reorthogonalization) is the honest fix at this scale.

Lanczos converging from the outside in is a problem whenever the physics wants the
bottom of the spectrum, because "outside" is measured against the whole spectral
width: for the Laplacian above $\lambda_{\min}$ and its neighbour are separated by
under a hundredth of the range, while $\lambda_{\max}$ sits alone at the far edge.
**Shift-invert** repairs this by running the iteration on a different operator,
since the eigenvalues of $(A-\sigma I)^{-1}$ are

```{math}
:label: eq-shiftinvert
\mu_i = \frac{1}{\lambda_i - \sigma},
```

so whichever $\lambda_i$ lie nearest the shift $\sigma$ become the largest $\mu_i$
by a wide margin, and the outside-in bias now points where we want it. Taking
$\sigma=0$ targets the smallest eigenvalues, which is what
`eigsh(A_rect, k=8, sigma=0.0)` means in
[§3.9](../03-electrodynamics/waveguides-cavities.ipynb). The price is one sparse
factorization of $A-\sigma I$, formed once and reused at every iteration: a real
cost, and still nothing beside a dense $O(N^3)$ eigendecomposition. The
alternative idiom `which="SA"` skips the factorization and asks for the smallest
algebraic eigenvalues directly, which is what
[§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb) and
[§8.2](../08-electronic-structure/exact-laboratory.ipynb) use; it reaches the same
answer through more iterations, and the timings below measure the difference.

1. Write `lanczos(A, v0, k)` implementing {eq}`eq-lanczos` and returning the
   diagonal `alpha` (length $k$) and off-diagonal `beta` (length $k-1$) of $T_k$.
   The operator must be touched only through `A @ q`; reorthogonalize each new
   $\mathbf w_j$ against the whole stored basis before normalizing.
   **Write this one yourself** — the implementation is the lesson.
2. Run it on the `laplacian_2d(20)` operator of Exercise 5 from the starting
   vector `numpy.random.default_rng(0).standard_normal(400)`, for
   $k=5,10,20,30,40$. Assemble $T_k$ with `numpy.diag` on the three bands, take
   `scipy.linalg.eigvalsh(T_k)`, and track three relative errors against
   Exercise 5's dense spectrum: the extreme Ritz values against $\lambda_{\min}$
   and $\lambda_{\max}$, and the closest Ritz value to the median eigenvalue
   $\lambda_{N/2}$ ({numref}`fig-eigsvd-lanczos`).
3. Take the six smallest eigenvalues twice, by `eigsh(A, k=6, which="SA")` and by
   shift-invert `eigsh(A, k=6, sigma=0.0)`, and check both against the first six
   of the dense spectrum. Then quantify {eq}`eq-shiftinvert`: compare the relative
   separation of the target eigenvalue,
   $(\lambda_2-\lambda_1)/(\lambda_N-\lambda_1)$, with the same quantity for the
   $\mu_i=1/\lambda_i$.
4. Time all three routes (dense `eigvalsh` on `.toarray()`, `which="SA"`, and
   shift-invert) for $n=20,30,40,50,60$ with `time.perf_counter`, and plot the
   wall-clock cost against $N$ with an $N^3$ reference slope
   ({numref}`fig-eigsvd-eig-timing`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    err_min[-1] < 1e-3 and err_max[-1] < 1e-3 and err_mid[-1] > 1e-2,
    "Lanczos converges from the outside in: at k = 40 both spectral extremes are "
    "pinned while the middle of the spectrum is not",
    f"rel. error λ_min {err_min[-1]:.1e}, λ_max {err_max[-1]:.1e}, "
    f"median {err_mid[-1]:.1e}",
)
validate.close(
    w_si,
    lam_dense[:6],
    "shift-invert eigsh(σ=0) returns the six smallest eigenvalues of the dense solve",
    rtol=1e-8,
)
validate.close(
    w_sa,
    lam_dense[:6],
    "eigsh(which='SA') reaches the same six without a factorization",
    rtol=1e-8,
)
validate.check(
    gap_invert / gap_direct > 10,
    "shift-invert widens the target eigenvalue's relative separation by an order of "
    "magnitude or more, which is why it converges",
    f"{gap_direct:.4f} → {gap_invert:.4f}  ({gap_invert / gap_direct:.0f}×)",
)
validate.check(
    t_dense[-1] > 5 * t_si[-1],
    "at N = 3600 the dense eigendecomposition costs several times the sparse "
    "shift-invert solve (a wall-clock check: the margin, not the number, is the claim)",
    f"dense {t_dense[-1]:.3f} s vs σ=0 {t_si[-1]:.3f} s ({t_dense[-1] / t_si[-1]:.0f}×)",
)

## Exercise 7 — The SVD: definition and geometry (worked animation)

The SVD, {eq}`eq-svd`, applies to any matrix, and its content is geometric. We
take two explicit matrices. First, to check the algebra, the $3\times2$

$$
A = \begin{bmatrix} 3 & 1 \\ 1 & 3 \\ 2 & 2 \end{bmatrix}:
$$

1. Compute $U,\Sigma,V^\top$ with `scipy.linalg.svd`, verify the reconstruction
   $A=U\Sigma V^\top$, and confirm $\sigma_i=\sqrt{\lambda_i(A^\top A)}$
   (`scipy.linalg.eigvalsh` on $A^\top A$).

Then, to *see* the geometry, take the $2\times2$

$$
B = \begin{bmatrix} 3 & 1 \\ 0 & 2 \end{bmatrix},
$$

which sends the unit circle to an ellipse.

2. Animate the map decomposed into its three SVD acts ($V^\top$ rotates,
   $\Sigma$ stretches along the axes, $U$ rotates) with `FuncAnimation`.
3. Confirm the resulting ellipse has semi-axes exactly $\sigma_1,\sigma_2$
   ({numref}`fig-eigsvd-ellipse`) — the claim the validation checks against the
   animated points.

In [ ]:
# (solution hidden on the public site)


### Validation 7a — the SVD algebra

In [ ]:
validate.close(U7 @ np.diag(s7) @ Vt7, A7, "A = UΣVᵀ reconstructs A", atol=1e-10)
validate.close(s7, sigma_from_AtA, "σ = √eig(AᵀA)", atol=1e-10)

Now the geometry of $B=\begin{bmatrix}3&1\\0&2\end{bmatrix}$. Fix the SVD sign
freedom so both $U$ and $V^\top$ are proper rotations, then animate the unit
circle through the three acts.

In [ ]:
# (solution hidden on the public site)


### Validation 7b — the geometry of the data

In [ ]:
validate.close(
    np.sort(ellipse_semi_axes)[::-1],
    np.sort(sb)[::-1],
    "the unit circle maps to an ellipse with semi-axes σᵢ",
    rtol=1e-3,
)
# (rtol reflects measuring the semi-axes as the max/min radius over a finite
# sampling of the mapped ellipse: a discretization, not a physics, tolerance.)

## Exercise 8 — Low-rank approximation and Eckart–Young

Here is the centrepiece. Eckart–Young, {eq}`eq-eckart`, says the truncated SVD is
the *best* rank-$k$ approximation and that its spectral-norm error is exactly the
first singular value we discarded, $\sigma_{k+1}$. We test this on a fixed,
reproducible matrix: **the $8\times6$ matrix $B$ whose entries are
`numpy.random.default_rng(0).standard_normal((8, 6))`** (seed 0, shape $8\times6$,
stated so the result is unambiguous).

1. For $k=1,2,3,4$ form the rank-$k$ truncation
   $\sum_{i\le k}\sigma_i\mathbf u_i\mathbf v_i^\top$ from the
   `scipy.linalg.svd` factors.
2. Confirm its spectral-norm error (`numpy.linalg.norm(..., 2)`) equals
   $\sigma_{k+1}$ ({numref}`fig-eigsvd-eckart`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    errors_k,
    sigma_next,
    "the rank-k truncated-SVD error equals σ_(k+1) (Eckart–Young)",
    rtol=1e-6,
)

## Exercise 9 — Compression: a low-rank matrix in noise (student exercise)

Real data is rarely exactly low-rank, but it is often *nearly* so, and the SVD
exposes that at a glance. The matrix to analyse is the explicit

$$
M_{ij} = \sin(2\pi x_i) + x_i\,g_j + 0.01\,\eta_{ij},
$$

with $x=\texttt{linspace}(0,1,40)$, $g=\texttt{linspace}(1,2,30)$, and noise
$\eta=\texttt{default\_rng(0).standard\_normal((40,30))}$ (seed 0, shape
$40\times30$). The first two terms are each a rank-1 outer product, so $M$ is
**rank 2 plus a small noise floor**.

1. Build $M$ from those three pieces: the two rank-1 outer products and the
   scaled noise.
2. Compute the singular values with `scipy.linalg.svd` and see two dominate
   before a cliff.
3. Reconstruct the rank-2 matrix from the truncated factors (broadcast the top
   two singular values into the product) and check the noise is gone.
4. Quantify $\sigma_3/\sigma_1$ as the effective-rank signal.

There is no animation here: this is a spectrum-and-reconstruction analysis, not
motion; a ✗ points at the construction of $M$ or the truncation.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 9

In [ ]:
validate.check(
    s9[2] / s9[0] < 0.05,
    "the matrix is effectively rank 2: the noise lives in the tiny tail of the spectrum",
    f"σ₃/σ₁ = {s9[2] / s9[0]:.4f}",
)

## Exercise 10 — The SVD → tensor networks (synthesis, and an honest horizon)

The move just made in Exercise 9 (keep the few large singular values, discard
the small tail) is, almost word for word, how the **tensor networks** of
many-body quantum physics are built. A quantum state of $N$ particles lives in a
Hilbert space of dimension $2^N$, hopelessly large; but a *physical* state
(a ground state of a local Hamiltonian) has, across any bipartition, an
**entanglement spectrum** (the singular values of the state reshaped into a
matrix) that decays rapidly, just like the noisy matrix above. A
**matrix-product state (MPS)** exploits exactly this: truncate each bond's SVD to
the largest few singular values (the "bond dimension"), and the $2^N$ cost
collapses to something linear in $N$. **DMRG** is the algorithm that does this
truncation variationally to find ground states.

No notebook in this course builds an MPS, and it is worth being exact about where
the course does stop. [§7.19](../07-quantum-statistical-mechanics/transverse-field-ising.ipynb)
measures the entanglement entropy of a spin chain and finds it saturating, which
is the evidence that a small bond dimension can suffice;
[§8.13](../08-electronic-structure/hubbard-model.ipynb) diagonalizes many-body
Hamiltonians exactly and names DMRG among the methods that reach the sizes exact
diagonalization cannot. Both point outward, neither implements. For the algorithms
themselves the standard entry point is Schollwöck's review of DMRG in the language
of matrix product states. What belongs here is the principle they rest on.
Keeping just the top two singular triples of the Exercise-9 matrix already
captures essentially all of its Frobenius norm: the same statement, for our toy
matrix, as "a low-entanglement state is well approximated by a small bond
dimension."

In [ ]:
# (solution hidden on the public site)


### Validation 10

In [ ]:
validate.check(
    captured > 0.99,
    "keeping a few singular values captures almost all of the matrix — the "
    "principle behind tensor-network (MPS/DMRG) compression",
    f"captured fraction = {captured:.4f}",
)

## Notebook summary

- The symmetric eigenproblem and the **spectral theorem** ($A=V\Lambda V^\top$, $V^\top V=I$),
  why the characteristic polynomial is the wrong tool, and the **QR algorithm** built from
  scratch; the generalized eigenproblem (callback to
  [§2.7](../02-classical-mechanics/small-oscillations.ipynb)).
- **Sparsity and Krylov methods.** The five-point Laplacian on a $20\times20$ grid has
  $5n^2-4n=1920$ non-zeros in $N^2=160{,}000$ entries, a density of $0.012$; at $n=320$
  the dense array would cost 84 GB against 6.5 MB stored sparsely. A hand-written
  **Lanczos** iteration, which touches the matrix only through $A\mathbf x$, pins both
  spectral extremes to a relative $10^{-4}$ by Krylov dimension $k=40$ out of $N=400$
  while the middle of the spectrum is still wrong in the second digit, and
  **shift-invert** turns that outside-in bias toward the smallest eigenvalues by
  widening their relative separation from $0.008$ to $0.60$, a factor of 72.
- The **SVD** ($A=U\Sigma V^\top$, $\sigma=\sqrt{\mathrm{eig}(A^\top A)}$), low-rank
  approximation and the Eckart–Young theorem, compression in noise, and the SVD as the seed of
  tensor-network methods.

## Outlook

- **Non-symmetric eigenproblems.** Complex spectra, non-orthogonal eigenvectors,
  and defective matrices with a Jordan form: numerically delicate, and a reason
  the symmetric case is so prized.
- **Power and inverse iteration.** The one-vector ancestors of Lanczos, which keep
  only the newest $A^k\mathbf b$ instead of the whole Krylov subspace and converge
  correspondingly slower; the seed of PageRank.
- **Arnoldi, and life without symmetry.** Drop $A=A^\top$ and the three-term
  recurrence becomes a full Hessenberg one: that is `scipy.sparse.linalg.eigs`,
  with complex Ritz values and no guarantee of an orthonormal eigenbasis.
- **Preconditioning.** Shift-invert is one way to reshape a spectrum in favour of
  the eigenvalues one wants; LOBPCG and preconditioned conjugate gradient (the
  iterative solvers named in the Outlook of [§0.4](linear-systems.ipynb)) are the
  others, and they are what make the largest ground-state problems move at all.
- **PCA is the SVD of a data matrix**: a forward link to the least-squares
  fitting of [§0.8](fitting-least-squares.ipynb), where the SVD also gives the
  pseudoinverse for rank-deficient problems.
- **Tensor networks (MPS/DMRG)** take the Eckart–Young truncation to many-body
  quantum states. This course goes as far as the evidence for them and stops
  there: no notebook builds an MPS or runs DMRG (outward, named — Schollwöck's
  review is the way in).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()